# Build the Badini OmniVoice offline browser bundle

This is a one-time model conversion job, **not an inference server**. It converts the exact fine-tuned Badini checkpoint into four ONNX graphs, pre-encodes the bundled Hayfa/Abdullah/Bafreen reference recordings, then dynamically quantizes the LLM's constant MatMul weights to INT8 with standard ONNX DynamicQuantizeLinear + MatMulInteger operators for the Safari WASM runtime. Speech generation after import runs locally in Safari.

Use a Colab **High-RAM** runtime if available. Export uses CPU PyTorch and may take a while. The Safari-compatible INT8 conversion is checked against the FP32 graph before it replaces the LLM. No public server, API endpoint or cloud inference is created.

In [ ]:
!rm -rf /content/badini-omnivoice-web
!git clone --depth 1 https://github.com/1alind/badini-omnivoice-web.git /content/badini-omnivoice-web
%cd /content/badini-omnivoice-web
!pip install -q -r requirements-build.txt


In [ ]:
from google.colab import files
from pathlib import Path
import shutil, zipfile

MODEL_ID = "Dezheen89/omnivoice-badini"
OUT = Path("/content/badini-model-bundle")
WORK = Path("/content/badini-export-work")
for folder in (OUT, WORK):
    if folder.exists(): shutil.rmtree(folder)
    folder.mkdir(parents=True)
print("Fine-tuned model:", MODEL_ID)
print("Output folder:", OUT)


In [ ]:
%cd /content/badini-omnivoice-web
!python tools/export_onnx.py --model-id "$MODEL_ID" --out "$OUT" --work "$WORK"


In [ ]:
%cd /content/badini-omnivoice-web
!python tools/quantize_mobile_int8.py --work-dir "$WORK" --release-dir "$OUT" --bits 8


In [ ]:
%cd /content/badini-omnivoice-web
!python tools/validate_bundle.py "$OUT"


## Download the model bundle

The ZIP contains the mobile INT8 LLM, the remaining FP32 ONNX components, tokenizer, manifest and pre-encoded speaker prompts. The dynamic INT8 conversion stores LLM MatMul weights as int8 and uses standard ONNX operators supported by the browser's WASM runtime. Extract the ZIP in the Files app and select all extracted files in Badini Voice. Do not upload model weights to GitHub.

In [ ]:
zip_path = "/content/badini-omnivoice-model.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_STORED) as z:
    for p in OUT.iterdir():
        if p.is_file(): z.write(p, arcname=p.name)
print("ZIP size (GB):", round(Path(zip_path).stat().st_size / 1e9, 2))
files.download(zip_path)
